# NYC Rideshare Profit Zone Recommender — Data Pipeline

This notebook covers the full data pipeline for the NYC Rideshare Profit Zone Recommender project — from raw data acquisition through to the final curated feature set used for modelling.

NYC's 2025 Congestion Pricing Program shifted FHV demand in ways that made traditional hotspot intuition unreliable. This project builds a zone-hour level profit score and trains a model to recommend the Top-5 most profitable zones for drivers each hour.

**Data sources used:**

- NYC TLC High-Volume FHV Trip Records (~120M trips/year)
- NOAA Central Park hourly weather data
- US public holiday calendar (`holidays` library)
- NYC TLC taxi zone shapefiles (263 zones)

**Train/validation/test split:**

- Train: Jan–Apr 2024
- Validation: May–Jun 2024
- Test: Jan–Jun 2025 (unseen year, post congestion pricing)

---


## 1. Data Acquisition

All trip data is sourced directly from the [NYC TLC Trip Record Data portal](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page). We focus exclusively on High-Volume FHV records (Uber, Lyft, Via) rather than yellow or green taxi data, as HVFHV trips dominate modern rideshare demand in NYC.

We download January–June for both 2024 and 2025. Using the same calendar months across both years controls for seasonal effects — ensuring that model generalisation to 2025 reflects the real policy shift rather than differences in summer vs winter demand.


### 1.1 Environment Setup


In [1]:
import os
import sys
import pathlib

PROJECT_ROOT = pathlib.Path(os.getcwd()).parent  # one level up from notebooks/
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
EXT_DIR = PROJECT_ROOT / "data" / "external"

sys.path.append(str(SCRIPTS_DIR))

### 1.2 Downloading Trip & Zone Data

The `tlc_io` helper script (in `/scripts`) handles idempotent downloads — it skips files that already exist locally and logs a manifest of what was fetched. This keeps the pipeline re-runnable without re-downloading ~6GB of data each time.


In [ ]:
from tlc_io import download_zones, download_trips

# Download taxi zone lookup CSV and shapefiles (used later for geospatial joins)
download_zones(unzip=True)

# Download HVFHV trip records for Jan–Jun 2024 and 2025
YEARS = ['2024', '2025']
MONTHS = list(range(1, 7))   # January through June
TYPES = ['fhv_hv']          # High-Volume FHV only

trip_results = download_trips(types=TYPES, years=YEARS, months=MONTHS)

# Summary: how many files downloaded and their sizes
print(f"Total files: {len(trip_results)}")
for r in trip_results:
    size_mb = r['size_bytes'] / 1e6
    print(f"  {r['year']}-{r['month']:>2}  {size_mb:>7.1f} MB  [{r['status']}]")

## 2. Trip Data Processing (HVFHV)

This section processes the raw HVFHV trip records into a clean **zone × hour** feature table ready for modelling.

The core challenge here is scale and quality. Each year contains ~120 million raw trip records, many of which contain implausible values — negative durations, impossibly fast speeds, or fare amounts that suggest recording errors rather than real trips. A multi-stage filtering pipeline removes these while preserving operationally meaningful edge cases like airport surcharges and storm-hour demand spikes.

The output of this section is a single aggregated table at the **zone–hour level**, with lag features and a custom profit score — the modelling target.


### 2.1 Spark Session

PySpark is used throughout to handle the volume of data. Key configurations:

- `spark.driver.memory = 8g` — required for aggregations over ~120M rows
- `spark.sql.session.timeZone = America/New_York` — ensures all timestamps are interpreted in NYC local time, not UTC, which matters for hour-of-day features
- Adaptive query execution enabled to handle partition skew during zone-level aggregations


In [ ]:
from scripts.spark_session import get_spark
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

spark = get_spark(app_name="01_process_fhvhv")

### 2.2 Schema Normalisation

The 2024 and 2025 datasets have a minor schema difference: the `cbd_congestion_fee` column (introduced with the Congestion Pricing Program) is present in 2025 records but absent in 2024. To allow both years to be processed through the same pipeline, we backfill this column with `0.0` for 2024 records.

Y/N flag columns (e.g. `shared_request_flag`, `wav_request_flag`) are also cast to boolean for consistency.


In [ ]:
def lowercase_cols(df):
    return df.toDF(*[c.lower() for c in df.columns])


def ny_to_bool_col(col):
    """Convert TLC Y/N string flags to boolean."""
    return (F.when(F.col(col) == "Y", True)
             .when(F.col(col) == "N", False)
             .otherwise(None))


def normalize_fhvhv(df, year: int):
    """Lowercase columns, backfill cbd_congestion_fee for 2024, cast Y/N flags to boolean."""
    df = lowercase_cols(df)

    # 2024 data predates the Congestion Pricing Program — backfill with 0.0
    if year == 2024 and "cbd_congestion_fee" not in df.columns:
        df = df.withColumn("cbd_congestion_fee", F.lit(0.0).cast("double"))

    flags = ["shared_request_flag", "shared_match_flag",
             "access_a_ride_flag", "wav_request_flag", "wav_match_flag"]
    for f in flags:
        if f in df.columns:
            df = df.withColumn(f, ny_to_bool_col(f))

    return df

### 2.3 Feature Derivation

Three derived features are computed from the raw timestamps and distance fields:

- **`trip_minutes`** — trip duration in minutes. Derived from dropoff minus pickup timestamps where available; falls back to the raw `trip_time` field (seconds ÷ 60) otherwise.
- **`pickup_hour_local`** — the NYC-local hour bucket for each trip, used as the aggregation key. Using local time (not UTC) is critical — a trip at midnight NYC time is UTC-4 or UTC-5 depending on daylight saving, and misassigning it would corrupt the hour-of-day signal.
- **`avg_speed_mph`** — derived as `(trip_miles / trip_minutes) × 60`. Nulled out when duration is zero to avoid division errors.


In [ ]:
def add_time_columns_local(df):
    """Cast to NYC-local timestamps and derive pickup hour bucket."""
    if "pickup_datetime" in df.columns:
        df = df.withColumn("pickup_ts_local",
                           F.col("pickup_datetime").cast("timestamp"))
        df = df.withColumn("pickup_hour_local",
                           F.date_trunc("hour", F.col("pickup_ts_local")))
    if "dropoff_datetime" in df.columns:
        df = df.withColumn("dropoff_ts_local",
                           F.col("dropoff_datetime").cast("timestamp"))
    return df


def add_trip_minutes(df):
    """Derive trip duration in minutes from timestamps or fallback trip_time field."""
    if "pickup_ts_local" in df.columns and "dropoff_ts_local" in df.columns:
        df = df.withColumn(
            "trip_minutes",
            (F.col("dropoff_ts_local").cast("long") -
             F.col("pickup_ts_local").cast("long")) / 60.0
        )
    elif "trip_time" in df.columns:
        df = df.withColumn("trip_minutes", (F.col("trip_time") / 60.0))
    else:
        df = df.withColumn("trip_minutes", F.lit(None).cast("double"))
    return df


def add_trip_avg_speed(df, miles_col="trip_miles",
                       minutes_col="trip_minutes", out_col="avg_speed_mph"):
    """Compute average speed in mph. Returns null when duration is zero or missing."""
    if miles_col in df.columns and minutes_col in df.columns:
        df = df.withColumn(
            out_col,
            F.when(
                F.col(minutes_col).isNotNull() &
                (F.col(minutes_col) > 0) &
                F.col(miles_col).isNotNull(),
                F.col(miles_col) * F.lit(60.0) / F.col(minutes_col)
            ).otherwise(F.lit(None).cast("double"))
        )
    return df

### 2.4 Fare Column Handling

Fare-related columns occasionally contain nulls (e.g. missing airport fee when no airport was involved). These are filled with `0.0` rather than dropped, since a null airport fee is semantically zero — not a missing observation. Dropping these rows would incorrectly discard valid trips.


In [ ]:
def fill_amount_nulls_with_zero(df):
    """Cast fare/fee columns to double and fill nulls with 0.0."""
    amount_cols = [
        "base_passenger_fare", "tolls", "bcf", "sales_tax",
        "congestion_surcharge", "airport_fee", "tips",
        "driver_pay", "cbd_congestion_fee"
    ]
    for c in amount_cols:
        if c in df.columns:
            df = df.withColumn(c, F.coalesce(
                F.col(c).cast("double"), F.lit(0.0)))
    return df

### 2.5 Data Quality Filtering

Filtering happens in two stages:

**Stage 1 — Fatal filters (hard rules):** Records that are physically impossible or violate TLC data integrity rules are removed unconditionally. Thresholds are set conservatively to avoid discarding legitimate edge cases.

| Rule                | Threshold                           | Rationale                                                                          |
| ------------------- | ----------------------------------- | ---------------------------------------------------------------------------------- |
| Trip duration       | 2 – 240 min                         | Sub-2-min trips are likely app errors; 240 min (4hr) is implausible within NYC     |
| Trip distance       | 0.05 – 100 miles                    | Near-zero distances indicate failed GPS; 100 miles exceeds NYC boundaries          |
| Average speed       | < 80 mph; < 1 mph if trip > 10 min  | 80 mph is impossible on NYC surface roads; crawling trips indicate stalled records |
| Timestamp integrity | Dropoff must be after pickup        | Negative durations are data errors                                                 |
| Zone validity       | Location IDs must be in range 1–263 | TLC defines exactly 263 zones                                                      |
| Fare integrity      | No negative fare components         | TLC rules prohibit negative base fares, tolls, or taxes                            |

**Stage 2 — IQR outlier removal:** After fatal filtering, a monthly IQR filter is applied to fare amounts per zone. This removes statistical outliers while deliberately preserving high but plausible fares (e.g. JFK airport trips with surcharges). The filter is applied per month rather than globally to account for seasonal fare variation.


In [ ]:
# Filtering thresholds
MAX_TRIP_MINUTES = 240.0   # 4 hours — implausible for NYC
MIN_TRIP_MINUTES = 2.0     # below this is likely an app/recording error
MAX_TRIP_MILES = 100.0   # exceeds NYC boundaries
MIN_TRIP_MILES = 0.05    # near-zero distance = GPS failure
MAX_SPEED_MPH = 80.0    # impossible on surface roads
MIN_SPEED_MPH = 1.0     # crawling — only flagged if trip > 10 min
LOW_SPEED_MINLEN = 10.0    # minutes threshold for low-speed check


def build_violation_exprs(df):
    """Returns a dict of rule_name -> Boolean Column (True = violation to remove)."""
    exprs = {}

    # Duration rules
    if "trip_minutes" in df.columns:
        exprs["trip_minutes_neg"] = F.col("trip_minutes") < 0
        exprs["trip_minutes_too_short"] = (
            F.col("trip_minutes") < MIN_TRIP_MINUTES) & F.col("trip_minutes").isNotNull()
        exprs["trip_minutes_too_long"] = F.col(
            "trip_minutes") > MAX_TRIP_MINUTES

    # Distance rules
    if "trip_miles" in df.columns:
        exprs["trip_miles_neg"] = F.col("trip_miles") < 0
        exprs["trip_miles_too_short"] = (
            F.col("trip_miles") < MIN_TRIP_MILES) & F.col("trip_miles").isNotNull()
        exprs["trip_miles_too_long"] = F.col("trip_miles") > MAX_TRIP_MILES

    # Speed rules
    if "avg_speed_mph" in df.columns:
        exprs["speed_too_fast"] = F.col("avg_speed_mph") > MAX_SPEED_MPH
        exprs["speed_too_slow"] = (
            (F.col("avg_speed_mph") < MIN_SPEED_MPH) &
            (F.col("trip_minutes") > LOW_SPEED_MINLEN)
        )

    # Timestamp integrity
    if "pickup_ts_local" in df.columns and "dropoff_ts_local" in df.columns:
        exprs["dropoff_before_pickup"] = F.col(
            "dropoff_ts_local") <= F.col("pickup_ts_local")

    # Zone validity (TLC defines zones 1–263)
    for loc_col in ["pulocationid", "dolocationid"]:
        if loc_col in df.columns:
            exprs[f"{loc_col}_invalid"] = (
                F.col(loc_col).isNull() |
                (F.col(loc_col) < 1) |
                (F.col(loc_col) > 263)
            )

    # Fare integrity — TLC rules prohibit negative fare components
    for fare_col in ["base_passenger_fare", "tolls", "sales_tax", "driver_pay"]:
        if fare_col in df.columns:
            exprs[f"{fare_col}_neg"] = F.col(fare_col) < 0

    return exprs


def apply_fatal_filter(df):
    """Remove records violating any hard quality rule. Returns filtered df."""
    violation_exprs = build_violation_exprs(df)
    if not violation_exprs:
        return df
    combined_violation = reduce(lambda a, b: a | b, violation_exprs.values())
    return df.filter(~combined_violation)


def apply_iqr_filter(df, fare_col="base_passenger_fare", group_col="month"):
    """
    Monthly IQR filter on fare amounts.
    Removes statistical outliers while preserving high-but-plausible fares
    (e.g. airport trips with surcharges). Applied per month to account for
    seasonal fare variation.
    """
    quantiles = df.groupBy(group_col).agg(
        F.percentile_approx(fare_col, 0.25).alias("q1"),
        F.percentile_approx(fare_col, 0.75).alias("q3")
    ).withColumn("iqr", F.col("q3") - F.col("q1"))

    df = df.join(quantiles, on=group_col, how="left")
    df = df.filter(
        F.col(fare_col).between(
            F.col("q1") - 1.5 * F.col("iqr"),
            F.col("q3") + 1.5 * F.col("iqr")
        )
    ).drop("q1", "q3", "iqr")
    return df

### 2.6 Zone–Hour Aggregation & Profit Score

Individual trip records are aggregated to the **zone × hour** level — the unit of analysis for modelling. For each combination of pickup zone and hour, we compute:

- `n_trips` — total trip count (volume signal)
- `avg_duration`, `avg_miles`, `avg_speed_mph` — trip characteristic averages
- `profit_score` — the modelling target (see below)

**Profit score definition:**

$$\text{profit\_score}_{z,h} = \frac{\text{total driver pay}_{z,h}}{\text{total trip minutes}_{z,h}} \times n\_trips_{z,h}$$

Tips are deliberately excluded from driver pay — they are unpredictable and not guaranteed. This produces a conservative but consistent measure of earnings potential that reflects what a driver can reliably expect, not a best-case scenario.

**Lag features** are added to capture temporal demand persistence:

- `n_trips_t_1` — trips in the previous hour (immediate continuity)
- `n_trips_t_24` — trips at the same hour yesterday (daily rhythm)

Both features are computed strictly from past observations — no data leakage. At inference time, both values are already known.


In [ ]:
from pyspark.sql import Window


def aggregate_to_zone_hour(df):
    """Aggregate cleaned trip records to zone x hour level with profit score."""
    agg = df.groupBy("pulocationid", "pickup_hour_local", "year", "month").agg(
        F.count("*").alias("n_trips"),
        F.sum("trip_minutes").alias("sum_trip_minutes"),
        F.sum("driver_pay").alias("sum_driver_revenue"),
        F.avg("trip_minutes").alias("avg_duration"),
        F.avg("trip_miles").alias("avg_miles"),
        F.avg("avg_speed_mph").alias("speed_mph")
    )

    # Profit score: revenue efficiency × demand volume
    agg = agg.withColumn(
        "avg_revenue_per_min",
        F.when(F.col("sum_trip_minutes") > 0,
               F.col("sum_driver_revenue") / F.col("sum_trip_minutes")
               ).otherwise(F.lit(None))
    ).withColumn(
        "profit_score",
        F.col("avg_revenue_per_min") * F.col("n_trips")
    )
    return agg


def add_lag_features(df):
    """
    Add temporal lag features for n_trips.
    Both lags use only past observations — no data leakage.
    """
    zone_window = Window.partitionBy(
        "pulocationid").orderBy("pickup_hour_local")

    df = df.withColumn("n_trips_t_1",
                       F.lag("n_trips", 1).over(zone_window))   # previous hour
    df = df.withColumn("n_trips_t_24",
                       F.lag("n_trips", 24).over(zone_window))  # same hour yesterday
    return df

### 2.7 Pipeline Execution

The full pipeline is run across all three splits. Each split reads raw parquet files, applies normalisation, feature derivation, fatal filtering, IQR filtering, aggregation, and lag computation — then writes the result to partitioned parquet.

**Note:** This cell takes approximately 30 minutes to run in full.


In [ ]:
RAW_DIR = "../data/raw/fhv_hv"
PROCESSED_DIR = "../data/processed/fhv_hv/zone_hour_features_splits"

TRAIN_MONTHS = [(2024, m) for m in range(1, 5)]              # Jan–Apr 2024
VAL_MONTHS = [(2024, m) for m in range(5, 7)]              # May–Jun 2024
TEST_MONTHS = [(2025, m) for m in range(1, 7)]              # Jan–Jun 2025


def process_split(split_name, year_months, do_counts=True):
    """Run the full pipeline for a given split and write to parquet."""
    monthly_dfs = []

    for year, month in year_months:
        path = f"{RAW_DIR}/fhvhv_tripdata_{year}-{month:02d}.parquet"
        print(f"  → {year}-{month:02d} | reading {path}")

        df = spark.read.parquet(path)
        df = normalize_fhvhv(df, year=year)
        df = add_time_columns_local(df)
        df = add_trip_minutes(df)
        df = add_trip_avg_speed(df)
        df = fill_amount_nulls_with_zero(df)

        if do_counts:
            n_raw = df.count()
            print(f"    rows raw = {n_raw:,}")

        df = apply_fatal_filter(df)

        if do_counts:
            n_after_fatal = df.count()
            print(
                f"    rows after fatal filter = {n_after_fatal:,} (dropped {n_raw - n_after_fatal:,})")

        df = apply_iqr_filter(df)

        if do_counts:
            n_after_iqr = df.count()
            print(
                f"    rows after IQR = {n_after_iqr:,} (dropped {n_after_fatal - n_after_iqr:,} this step)")

        df = df.withColumn("year", F.lit(year)).withColumn(
            "month", F.lit(month))
        monthly_dfs.append(aggregate_to_zone_hour(df))

    combined = reduce(lambda a, b: a.unionByName(b), monthly_dfs)
    combined = add_lag_features(combined)

    out_path = f"{PROCESSED_DIR}/{split_name}"
    combined.write.mode("overwrite").partitionBy(
        "year", "month").parquet(out_path)
    print(f" wrote split '{split_name}' → {out_path}")


process_split("train", TRAIN_MONTHS, do_counts=True)
process_split("val",   VAL_MONTHS,   do_counts=True)
process_split("test",  TEST_MONTHS,  do_counts=True)

### 2.8 Filtering Results

After applying both filtering stages across both years:

| Year     | Raw rows        | After filtering | Dropped        | Drop rate  |
| -------- | --------------- | --------------- | -------------- | ---------- |
| 2024     | 120,864,668     | 108,725,626     | 12,139,042     | 10.04%     |
| 2025     | 120,995,191     | 108,044,497     | 12,950,694     | 10.70%     |
| **Both** | **241,859,859** | **216,770,123** | **25,089,736** | **10.37%** |

The ~10% drop rate is consistent across both years, which is a good sign — it suggests the filtering rules are stable and not overly sensitive to the policy change between years. A dramatically different drop rate in 2025 would have indicated the congestion pricing shift was affecting data quality, not just demand patterns.


### 2.9 Output Schema Verification

A quick sanity check on the output feature table for the first month of the test set.


In [ ]:
base = "../data/processed/fhv_hv/zone_hour_features_splits/test"
df_check = (
    spark.read
    .option("basePath", base)
    .parquet(f"{base}/year=2025/month=1")
)
df_check.show(3, truncate=False)

+------------+-------------------+-------+----------------+------------------+------------+---------+-------------------+--------------+-----------+------------+----+-----+
|pulocationid|pickup_hour_local  |n_trips|sum_trip_minutes|sum_driver_revenue|avg_duration|avg_miles|avg_revenue_per_min|profit_score  |n_trips_t_1|n_trips_t_24|year|month|
+------------+-------------------+-------+----------------+------------------+------------+---------+-------------------+--------------+-----------+------------+----+-----+
|12          |2025-01-01 00:00:00|6      |97.4            |122.38            |16.23       |5.55     |1.256              |7.539         |NULL       |NULL        |2025|1    |
|12          |2025-01-01 02:00:00|1      |11.15           |21.1              |11.15       |2.99     |1.892              |1.892         |6          |NULL        |2025|1    |
|12          |2025-01-01 04:00:00|2      |58.1            |71.61             |29.05       |11.64    |1.233              |2.465         